# 📰 RAG Pipeline - Vietnamese News Assistant

**Pipeline nâng cao:**
```
User Prompt
    ↓ [1] transform_query()    → Tạo 5 queries đa dạng
    ↓ [2] retrieve_multi()     → Retrieve chunks theo nội dung (content-based)
    ↓ [3] filter_by_metadata() → Lọc CHẶT theo category phù hợp
    ↓ [4] get_top_k()          → Chọn K bài chính xác nhất từ pool đã lọc
    ↓ [5] LLM Generate         → Trả lời dựa trên bài báo
```

## 1. 📦 Cài đặt môi trường (Google Colab)

Chạy các cell dưới theo thứ tự: cài thư viện → cài Ollama → serve → pull model.

In [ ]:
# ── 1.1 Cài đặt Python packages ──────────────────────────────────────────────
%pip install -q chromadb sentence-transformers langchain langchain-ollama gradio
print("✅ Packages installed!")

In [ ]:
# ── 1.2 Cài đặt Ollama trên Google Colab ─────────────────────────────────────
# Cần cài zstd trước (dependency của Ollama)
!apt-get install -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed!")

In [ ]:
# ── 1.3 Khởi động Ollama server ───────────────────────────────────────────────
import subprocess
import time
import requests

# Chạy ollama serve trong background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Đợi server khởi động
print("⏳ Đang khởi động Ollama server...", end="")
for _ in range(30):
    time.sleep(1)
    try:
        resp = requests.get("http://localhost:11434", timeout=2)
        if resp.status_code == 200:
            print(f"\n✅ Ollama server đang chạy tại http://localhost:11434")
            break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n⚠️  Server chưa phản hồi — vẫn tiếp tục...")

In [ ]:
# ── 1.4 Pull model gemma3:4b ──────────────────────────────────────────────────
# Quá trình tải ~3-4 GB, mất khoảng 3-5 phút tùy kết nối
print("⏳ Đang pull gemma3:4b (có thể mất vài phút)...")
!ollama pull gemma3:4b

# Kiểm tra model đã có
print("\n📋 Danh sách models đã cài:")
!ollama list

## 2. 🔧 Import Libraries

In [ ]:
import chromadb
import numpy as np
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import gradio as gr
import json
import re

print("✅ Imports OK!")

## 3. 🔌 Kết nối ChromaDB

In [ ]:
# ============================================================
# CẤU HÌNH — CHỈ SỬA Ở ĐÂY
# ============================================================
CHROMA_URL        = "https://f118-14-186-205-192.ngrok-free.app"  # ← URL ngrok
CHROMA_AUTH_TOKEN = "test-token"
COLLECTION_NAME   = "news_articles"
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"
LLM_MODEL         = "gemma3:4b"  # ← Model vừa pull

# ============================================================
settings = Settings(
    anonymized_telemetry=False,
    chroma_client_auth_provider="chromadb.auth.token_authn.TokenAuthClientProvider",
    chroma_client_auth_credentials=CHROMA_AUTH_TOKEN,
)
client = chromadb.HttpClient(host=CHROMA_URL, port=443, ssl=True, settings=settings)
heartbeat = client.heartbeat()
print(f"✅ ChromaDB connected | Heartbeat: {heartbeat}")

collection = client.get_collection(name=COLLECTION_NAME)
total_docs = collection.count()
print(f"📊 Collection '{COLLECTION_NAME}': {total_docs} chunks")

## 4. 🧠 Load Models

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
print(f"✅ Embedding: {EMBEDDING_MODEL} | dim={embedding_model.get_sentence_embedding_dimension()}")

llm = ChatOllama(model=LLM_MODEL, temperature=0.1)
print(f"✅ LLM: {LLM_MODEL}")

# Cross-encoder để re-rank (Top-K)
cross_encoder = None
try:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    print("✅ Cross-encoder: ms-marco-MiniLM-L-6-v2")
except Exception as e:
    print(f"⚠️  Cross-encoder không khả dụng: {e}")

## 5. 🔄 Step 1 — `transform_query()`: Biến 1 prompt → 5 queries

Dùng LLM để tạo ra 5 câu truy vấn đa dạng từ prompt gốc của user.
Mỗi query khai thác một góc độ khác nhau → tăng recall khi vector search.

In [ ]:
def transform_query(user_prompt: str, n: int = 5) -> List[str]:
    """
    Biến user prompt thành N queries đa dạng bằng LLM.

    Kỹ thuật: Multi-Query Retrieval.
    Mỗi query khai thác một khía cạnh/từ ngữ khác nhau của cùng chủ đề
    → giúp vector search bao phủ nhiều bài báo liên quan hơn.

    Args:
        user_prompt: Câu hỏi gốc từ người dùng
        n: Số queries muốn tạo (mặc định 5)

    Returns:
        List các query strings (bao gồm cả query gốc)
    """
    system_msg = (
        "Bạn là chuyên gia tìm kiếm thông tin tin tức tiếng Việt. "
        "Nhiệm vụ của bạn là tạo ra các câu truy vấn tìm kiếm đa dạng "
        "để tìm bài báo liên quan đến yêu cầu của người dùng.\n\n"
        f"Từ câu hỏi gốc, hãy tạo ra đúng {n} câu truy vấn TÌM KIẾM khác nhau, "
        "mỗi câu khai thác một góc độ hoặc từ khóa khác nhau.\n\n"
        "YÊU CẦU:\n"
        "- Mỗi câu truy vấn ngắn gọn (5-15 từ)\n"
        "- Đa dạng về từ ngữ, không lặp ý\n"
        "- Phù hợp với ngữ cảnh báo chí tiếng Việt\n"
        "- Trả về ĐÚNG định dạng JSON sau, không thêm text khác:\n"
        '{"queries": ["query 1", "query 2", "query 3", "query 4", "query 5"]}'
    )

    messages = [
        SystemMessage(content=system_msg),
        HumanMessage(content=f"Câu hỏi gốc: {user_prompt}")
    ]

    try:
        response = llm.invoke(messages)
        raw = response.content.strip()

        # Parse JSON từ response (xử lý cả trường hợp LLM thêm text thừa)
        json_match = re.search(r'\{[^{}]*"queries"[^{}]*\}', raw, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            queries = data.get("queries", [])
        else:
            # Fallback: tách theo dòng nếu không có JSON
            lines = [l.strip().lstrip("0123456789.-) ") for l in raw.split("\n") if l.strip()]
            queries = [l for l in lines if len(l) > 5][:n]

    except Exception as e:
        print(f"⚠️  transform_query error: {e}")
        queries = []

    # Luôn bao gồm query gốc và đảm bảo không duplicate
    all_queries = [user_prompt]
    for q in queries:
        q = q.strip()
        if q and q.lower() != user_prompt.lower() and q not in all_queries:
            all_queries.append(q)

    # Nếu LLM không tạo đủ, thêm biến thể đơn giản
    if len(all_queries) < 2:
        all_queries.append(user_prompt + " tin tức")
        all_queries.append(user_prompt + " mới nhất")

    return all_queries[:n + 1]  # Trả về tối đa n+1 (gốc + n)


# --- Test ---
print("🔄 Test transform_query():")
queries = transform_query("tình hình kinh tế Việt Nam")
for i, q in enumerate(queries):
    print(f"  [{i}] {q}")

## 6. 🗃️ Step 2 — `retrieve_multi()`: Retrieve theo nội dung (content-based)

**Content-based retrieval** — mỗi chunk được đánh giá riêng:
1. Multi-query search → thu thập chunks ứng viên
2. **Re-score từng chunk** bằng cosine similarity với user prompt gốc
3. Chỉ giữ chunks có nội dung thực sự liên quan (trên ngưỡng)
4. Dedup theo `(link, chunk_id)` — không trùng
5. Group theo bài báo → mỗi bài chỉ chứa các phần liên quan

In [ ]:
def _cosine_similarity(vec_a, vec_b):
    """Tính cosine similarity giữa 2 vectors."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def retrieve_multi(
    queries: List[str],
    user_prompt: str,
    embedding_model,
    collection,
    max_fetch_per_query: int = 20,
    distance_threshold: float = 1.5,
    content_sim_threshold: float = 0.25,
) -> List[Dict]:
    """
    Retrieve chunks theo NỘI DUNG — content-based retrieval.

    Quy trình:
    1. Dùng multi-query để tìm chunks ứng viên từ ChromaDB
    2. Dedup theo (link, chunk_id) — cùng chunk không lấy 2 lần
    3. RE-SCORE: Tính cosine similarity giữa MỖI chunk và user prompt gốc
       → đảm bảo nội dung chunk thực sự liên quan đến câu hỏi
    4. Chỉ giữ chunks có content_similarity >= ngưỡng
    5. Group theo bài báo → mỗi bài chứa đúng các phần liên quan

    Args:
        queries: Danh sách queries đã transform
        user_prompt: Câu hỏi GỐC của user (dùng để re-score nội dung)
        embedding_model: SentenceTransformer
        collection: ChromaDB collection
        max_fetch_per_query: Số chunks tối đa mỗi query
        distance_threshold: Ngưỡng L2 distance cho multi-query search
        content_sim_threshold: Ngưỡng cosine similarity tối thiểu
            giữa chunk và user_prompt (0.0-1.0). Chunks dưới ngưỡng bị loại.
            Mặc định 0.25 (khá mở, phù hợp cross-lingual).

    Returns:
        List[Dict] các bài báo, mỗi bài chứa CHỈ các chunks
        có nội dung liên quan đến user prompt
    """
    # ── Bước 1: Multi-query search → thu thập chunks ứng viên ──
    # chunk_map: (link, chunk_id) → {text, meta, best_dist, queries}
    chunk_map: Dict[tuple, Dict] = {}

    for query_idx, query in enumerate(queries):
        embedding = embedding_model.encode(query, convert_to_numpy=True).tolist()

        try:
            results = collection.query(
                query_embeddings=[embedding],
                n_results=max_fetch_per_query,
                include=["documents", "metadatas", "distances"],
            )
        except Exception as e:
            print(f"⚠️  Query [{query_idx}] error: {e}")
            continue

        docs  = results["documents"][0]
        metas = results["metadatas"][0]
        dists = results["distances"][0]

        # Lọc theo distance threshold, fallback top-3
        relevant = [(i, d) for i, d in enumerate(dists) if d <= distance_threshold]
        if not relevant:
            relevant = [(i, dists[i]) for i in range(min(3, len(dists)))]

        for idx, dist in relevant:
            link     = metas[idx].get("link", f"unknown_{query_idx}_{idx}")
            chunk_id = metas[idx].get("chunk_id", idx)
            key = (link, chunk_id)

            if key not in chunk_map:
                chunk_map[key] = {
                    "text": docs[idx],
                    "meta": metas[idx],
                    "best_distance": dist,
                    "matched_queries": [query],
                }
            else:
                chunk_map[key]["best_distance"] = min(
                    chunk_map[key]["best_distance"], dist
                )
                if query not in chunk_map[key]["matched_queries"]:
                    chunk_map[key]["matched_queries"].append(query)

    if not chunk_map:
        return []

    # ── Bước 2: Re-score từng chunk với user prompt gốc ────────
    # Embed user prompt
    prompt_emb = embedding_model.encode(user_prompt, convert_to_numpy=True)

    # Embed tất cả chunk texts cùng lúc (batch, nhanh hơn)
    keys_list = list(chunk_map.keys())
    texts_list = [chunk_map[k]["text"] for k in keys_list]
    chunk_embs = embedding_model.encode(texts_list, convert_to_numpy=True)

    # Tính cosine similarity với user prompt cho từng chunk
    scored_chunks = []
    for i, key in enumerate(keys_list):
        sim = _cosine_similarity(prompt_emb, chunk_embs[i])
        chunk_map[key]["content_similarity"] = sim
        scored_chunks.append((key, sim))

    # ── Bước 3: Lọc chunks theo content similarity ─────────────
    filtered_keys = [key for key, sim in scored_chunks if sim >= content_sim_threshold]

    # Fallback: nếu không chunk nào đạt ngưỡng, lấy top-10 cao nhất
    if not filtered_keys:
        scored_chunks.sort(key=lambda x: x[1], reverse=True)
        filtered_keys = [key for key, _ in scored_chunks[:10]]

    # ── Bước 4: Group chunks theo bài báo ──────────────────────
    article_chunks: Dict[str, Dict] = {}
    for key in filtered_keys:
        link, chunk_id = key
        chunk_data = chunk_map[key]

        if link not in article_chunks:
            article_chunks[link] = {
                "meta": chunk_data["meta"],
                "best_distance": chunk_data["best_distance"],
                "best_similarity": chunk_data["content_similarity"],
                "matched_queries": list(chunk_data["matched_queries"]),
                "chunks": [],
            }
        else:
            article_chunks[link]["best_distance"] = min(
                article_chunks[link]["best_distance"],
                chunk_data["best_distance"]
            )
            article_chunks[link]["best_similarity"] = max(
                article_chunks[link]["best_similarity"],
                chunk_data["content_similarity"]
            )
            for q in chunk_data["matched_queries"]:
                if q not in article_chunks[link]["matched_queries"]:
                    article_chunks[link]["matched_queries"].append(q)

        article_chunks[link]["chunks"].append({
            "chunk_id": chunk_id,
            "text": chunk_data["text"],
            "distance": chunk_data["best_distance"],
            "content_similarity": chunk_data["content_similarity"],
        })

    # ── Bước 5: Build articles ─────────────────────────────────
    articles = []
    for link, data in article_chunks.items():
        meta = data["meta"]
        # Sort chunks theo chunk_id để giữ đúng thứ tự bài viết
        sorted_chunks = sorted(data["chunks"], key=lambda c: c["chunk_id"])
        content = "\n\n".join(c["text"] for c in sorted_chunks)

        articles.append({
            "title":              meta.get("title", "N/A"),
            "category":           meta.get("category", "N/A"),
            "published_date":     meta.get("published_date", "N/A"),
            "link":               link,
            "full_content":       content,
            "distance":           data["best_distance"],
            "content_similarity": data["best_similarity"],
            "matched_queries":    data["matched_queries"],
            "num_chunks":         len(sorted_chunks),
            "chunk_ids":          [c["chunk_id"] for c in sorted_chunks],
            "chunk_similarities": [round(c["content_similarity"], 3) for c in sorted_chunks],
        })

    # Sort bài báo theo content similarity cao nhất (giảm dần)
    articles.sort(key=lambda x: x["content_similarity"], reverse=True)
    return articles


# --- Test ---
print("🗃️  Test retrieve_multi():")
test_queries = ["tình hình kinh tế Việt Nam", "kinh tế vĩ mô Việt Nam"]
test_articles = retrieve_multi(
    test_queries,
    user_prompt="tình hình kinh tế Việt Nam",
    embedding_model=embedding_model,
    collection=collection,
)
print(f"✅ Tìm được {len(test_articles)} bài báo (chỉ chunks liên quan về nội dung)")
for a in test_articles[:5]:
    print(f"  [sim={a['content_similarity']:.3f}] [{a['category']}] {a['title']}")
    print(f"           {a['num_chunks']} chunks (IDs: {a['chunk_ids']}, sims: {a['chunk_similarities']})")

## 7. 🏷️ Step 3 — `filter_by_metadata()`: Lọc CHẶT bài báo theo category phù hợp

Dùng LLM để xác định **categories nào phù hợp** với prompt.
Sau đó **chỉ giữ lại bài thuộc đúng categories đó** → đưa vào `get_top_k()`.

In [ ]:
def determine_relevant_categories(user_prompt: str, available_categories: List[str]) -> List[str]:
    """
    Dùng LLM để xác định categories nào trong DB phù hợp với prompt của user.

    Args:
        user_prompt: Câu hỏi gốc từ người dùng
        available_categories: Danh sách categories thực tế trong DB

    Returns:
        List categories phù hợp (subset của available_categories)
    """
    if not available_categories:
        return []

    cats_str = ", ".join(f'"{c}"' for c in available_categories)

    system_msg = (
        "Bạn là hệ thống phân loại tin tức. "
        "Nhiệm vụ là chọn các danh mục (category) tin tức PHÙ HỢP NHẤT với câu hỏi của người dùng.\n\n"
        f"Các danh mục có sẵn trong database: {cats_str}\n\n"
        "Hãy chọn MỘT HOẶC NHIỀU danh mục phù hợp nhất. "
        "Nếu câu hỏi rất chung chung, có thể trả về tất cả.\n"
        "Trả về ĐÚNG định dạng JSON sau, không thêm text khác:\n"
        '{"relevant_categories": ["category1", "category2"]}'
    )

    messages = [
        SystemMessage(content=system_msg),
        HumanMessage(content=f"Câu hỏi người dùng: {user_prompt}")
    ]

    try:
        response = llm.invoke(messages)
        raw = response.content.strip()

        json_match = re.search(r'\{[^{}]*"relevant_categories"[^{}]*\}', raw, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            selected = data.get("relevant_categories", [])
            valid = [c for c in selected if c in available_categories]
            return valid if valid else available_categories
    except Exception as e:
        print(f"⚠️  determine_relevant_categories error: {e}")

    return available_categories  # Fallback


def filter_by_metadata(
    articles: List[Dict],
    user_prompt: str,
) -> Tuple[List[Dict], List[str]]:
    """
    Lọc CHẶT bài báo — chỉ giữ bài thuộc category phù hợp với prompt.

    Quy trình:
    1. Lấy danh sách categories từ danh sách bài báo hiện có
    2. Dùng LLM xác định categories nào phù hợp với prompt
    3. Chỉ giữ lại bài thuộc đúng category đó (strict)
    4. Fallback: giữ tất cả nếu không có bài nào match (edge case)

    Returns:
        (filtered_articles, relevant_categories)
        filtered_articles: CHỈ các bài đúng category — đưa thẳng vào get_top_k()
    """
    if not articles:
        return [], []

    available_categories = list(set(
        a["category"].strip()
        for a in articles
        if a.get("category") and a["category"].strip() not in ("", "N/A")
    ))

    if not available_categories:
        return articles, []

    relevant_cats = determine_relevant_categories(user_prompt, available_categories)

    # Lọc CHẶT: chỉ giữ bài thuộc category phù hợp
    matched = [a for a in articles if a.get("category", "").strip() in relevant_cats]

    # Fallback: nếu không có bài nào match → dùng tất cả để tránh trả về rỗng
    if not matched:
        print(f"⚠️  Không có bài nào thuộc categories {relevant_cats}. Fallback: dùng tất cả.")
        return articles, relevant_cats

    return matched, relevant_cats


# --- Test ---
print("🏷️  Test filter_by_metadata():")
filtered, cats = filter_by_metadata(test_articles, "tình hình kinh tế Việt Nam")
print(f"✅ Categories phù hợp: {cats}")
print(f"✅ Bài báo sau lọc (strict): {len(filtered)} bài (từ {len(test_articles)} bài ban đầu)")
for a in filtered:
    print(f"  [sim={a['content_similarity']:.3f}] [{a['category']}] {a['title']} ({a['num_chunks']} chunks)")

## 8. 🏆 Step 4 — `get_top_k()`: Chọn K bài chính xác nhất

Dùng **cross-encoder re-ranking** để chấm điểm từng bài báo với query gốc,
sau đó chọn K bài có điểm cao nhất từ **pool đã được lọc category**.

In [ ]:
def get_top_k(
    articles: List[Dict],
    user_prompt: str,
    k: int = 5,
    cross_encoder=None,
) -> List[Dict]:
    """
    Chọn K bài báo chính xác nhất bằng cross-encoder re-ranking.

    Input là pool bài báo ĐÃ được lọc chặt theo category từ filter_by_metadata().

    Cross-encoder chấm điểm từng cặp (query, full_article_content)
    chính xác hơn nhiều so với bi-encoder (vector cosine similarity),
    vì nó xem xét ngữ cảnh đầy đủ của cả query lẫn document.

    Args:
        articles: Danh sách bài báo sau filter_by_metadata() — CHỈ bài đúng category
        user_prompt: Câu hỏi gốc của user
        k: Số bài báo muốn giữ lại
        cross_encoder: CrossEncoder model (None → fallback dùng content_similarity)

    Returns:
        Top-K bài báo, mỗi bài có thêm field 'relevance_score'
    """
    if not articles:
        return []

    if len(articles) <= k:
        for a in articles:
            a.setdefault("relevance_score", a.get("content_similarity", 0))
        return articles

    if cross_encoder is not None:
        MAX_CONTENT_LEN = 512
        pairs = [
            [user_prompt, a["full_content"][:MAX_CONTENT_LEN]]
            for a in articles
        ]
        scores = cross_encoder.predict(pairs)
        for i, a in enumerate(articles):
            a["relevance_score"] = float(scores[i])
        articles_scored = sorted(articles, key=lambda x: x["relevance_score"], reverse=True)
    else:
        # Fallback: dùng content_similarity đã tính ở retrieve_multi
        for a in articles:
            a["relevance_score"] = a.get("content_similarity", 0)
        articles_scored = sorted(articles, key=lambda x: x["relevance_score"], reverse=True)

    return articles_scored[:k]


# --- Test ---
print("🏆 Test get_top_k():")
top_k = get_top_k(filtered, "tình hình kinh tế Việt Nam", k=5, cross_encoder=cross_encoder)
print(f"✅ Top-{len(top_k)} bài báo chính xác nhất (từ {len(filtered)} bài đúng category):")
for i, a in enumerate(top_k):
    score = a.get('relevance_score', 0)
    print(f"  #{i+1} [score={score:.4f}] [{a['category']}] {a['title']} ({a['num_chunks']} chunks)")

## 9. 🤖 RAG Pipeline hoàn chỉnh

Kết hợp toàn bộ pipeline:
```
transform_query → retrieve_multi (content-based) → filter_by_metadata (strict) → get_top_k → LLM
```

Context + chỉ thị được đưa vào `HumanMessage` cuối cùng (tránh attention fade ở model nhỏ).

In [ ]:
# SystemMessage giữ ngắn gọn — chỉ định vai trò
SYSTEM_PROMPT_SHORT = "Bạn là trợ lý tin tức tiếng Việt. Bạn chỉ trả lời dựa trên dữ liệu được cung cấp."

# Chỉ thị chi tiết — sẽ đưa vào HumanMessage cùng context
RAG_INSTRUCTIONS = (
    "HƯỚNG DẪN BẮT BUỘC:\n"
    "1. NGHIÊM CẤM sử dụng kiến thức bên ngoài. CHỈ dùng thông tin từ các bài báo bên dưới.\n"
    "2. Nếu không có bài báo liên quan, trả lời: 'Không tìm thấy bài báo liên quan trong cơ sở dữ liệu.'\n"
    "3. Luôn trích dẫn tiêu đề bài báo và link nguồn khi trả lời.\n"
    "4. Trả lời bằng tiếng Việt, rõ ràng và súc tích.\n"
    "5. Nếu có nhiều bài báo liên quan, tổng hợp thông tin từ tất cả.\n"
    "6. KHÔNG ĐƯỢC bịa thêm thông tin ngoài những gì có trong các bài báo.\n"
)


def build_context_text(articles: List[Dict]) -> str:
    """Format danh sách bài báo thành context string cho LLM."""
    if not articles:
        return "[KHÔNG TÌM THẤY BÀI BÁO LIÊN QUAN]"
    parts = []
    for i, a in enumerate(articles):
        n_chunks = a.get('num_chunks', '?')
        parts.append(
            "BÀI BÁO " + str(i + 1) + " (" + str(n_chunks) + " phần liên quan)\n"
            + "Tiêu đề: " + a["title"] + "\n"
            + "Danh mục: " + a["category"] + "\n"
            + "Ngày đăng: " + a["published_date"] + "\n"
            + "Link: " + a["link"] + "\n"
            + "Nội dung:\n" + a["full_content"] + "\n"
        )
    return "\n".join(parts)


def rag_pipeline(
    user_query: str,
    chat_history: list = None,
    top_k: int = 5,
    n_queries: int = 5,
    distance_threshold: float = 1.5,
    content_sim_threshold: float = 0.25,
    debug: bool = False,
) -> str:
    """
    RAG Pipeline đầy đủ:
    1. transform_query    → Tạo N queries từ user prompt
    2. retrieve_multi     → Retrieve chunks theo nội dung (content-based)
    3. filter_by_metadata → Lọc CHẶT theo category
    4. get_top_k          → Chọn K bài chính xác nhất
    5. LLM Generate       → Trả lời dựa trên bài báo
    """
    if debug:
        print(f"\n{'='*60}")
        print(f"📝 User query: {user_query}")

    # ── STEP 1: Transform query ──────────────────────────────
    queries = transform_query(user_query, n=n_queries)
    if debug:
        print(f"\n🔄 Step 1 — Queries ({len(queries)}):")
        for i, q in enumerate(queries):
            print(f"  [{i}] {q}")

    # ── STEP 2: Retrieve (content-based, chunk-level) ────────
    articles = retrieve_multi(
        queries,
        user_prompt=user_query,
        embedding_model=embedding_model,
        collection=collection,
        max_fetch_per_query=20,
        distance_threshold=distance_threshold,
        content_sim_threshold=content_sim_threshold,
    )
    if debug:
        print(f"\n🗃️  Step 2 — Retrieved: {len(articles)} bài báo (content-based)")
        total_chunks = sum(a.get('num_chunks', 0) for a in articles)
        print(f"            Tổng chunks liên quan: {total_chunks}")
        for a in articles[:5]:
            print(f"              [sim={a['content_similarity']:.3f}] {a['title']} ({a['num_chunks']} chunks)")

    # ── STEP 3: Filter by metadata (luôn strict) ─────────────
    filtered, relevant_cats = filter_by_metadata(articles, user_query)
    if debug:
        print(f"\n🏷️  Step 3 — Relevant categories: {relevant_cats}")
        print(f"            After filter: {len(filtered)} bài (từ {len(articles)} bài)")

    # ── STEP 4: Top-K từ pool đã lọc category ────────────────
    top_articles = get_top_k(
        filtered, user_query, k=top_k, cross_encoder=cross_encoder
    )
    if debug:
        print(f"\n🏆 Step 4 — Top-{top_k}:")
        for a in top_articles:
            print(f"  [score={a.get('relevance_score', 0):.4f}] [{a['category']}] {a['title']} ({a['num_chunks']} chunks)")

    # ── STEP 5: Generate ─────────────────────────────────────
    context_text = build_context_text(top_articles)

    messages = [SystemMessage(content=SYSTEM_PROMPT_SHORT)]

    if chat_history:
        for human_msg, ai_msg in chat_history:
            messages.append(HumanMessage(content=str(human_msg)))
            messages.append(AIMessage(content=str(ai_msg)))

    final_prompt = (
        RAG_INSTRUCTIONS
        + "\n--- BẮT ĐẦU CÁC BÀI BÁO ---\n"
        + context_text
        + "\n--- KẾT THÚC CÁC BÀI BÁO ---\n\n"
        + "CÂU HỎI CỦA TÔI: " + user_query + "\n\n"
        + "Hãy trả lời DỰA HOÀN TOÀN vào các bài báo trên. "
        + "KHÔNG được dùng kiến thức bên ngoài."
    )
    messages.append(HumanMessage(content=final_prompt))

    if debug:
        print(f"\n📤 Final HumanMessage ({len(final_prompt)} chars)")
        print(f"   400 ký tự đầu: {final_prompt[:400]}")
        print(f"   ...")
        print(f"   200 ký tự cuối: {final_prompt[-200:]}")
        print(f"{'='*60}\n")

    response = llm.invoke(messages)
    return response.content


# === Test pipeline đầy đủ (debug=True) ===
print("🧪 Test full RAG pipeline (debug=True):")
answer = rag_pipeline(
    user_query="Tình hình của OpenAI",
    top_k=5,
    debug=True,
)
print(f"\n🤖 Trả lời:\n{answer}")

## 10. 🎨 Gradio Chat UI

In [ ]:
def chat_fn(message: str, history: list) -> str:
    """Gradio chat callback — dùng RAG pipeline đầy đủ."""
    try:
        chat_history = []
        if history:
            first = history[0]
            if isinstance(first, dict):
                i = 0
                while i < len(history) - 1:
                    h, a = history[i], history[i + 1]
                    if h.get("role") == "user" and a.get("role") == "assistant":
                        chat_history.append((h["content"], a["content"]))
                        i += 2
                    else:
                        i += 1
            elif isinstance(first, (list, tuple)) and len(first) == 2:
                chat_history = [(h, a) for h, a in history if h and a]

        return rag_pipeline(
            user_query=message,
            chat_history=chat_history,
            top_k=5,
            n_queries=5,
            distance_threshold=1.5,
            content_sim_threshold=0.25,
            debug=False,
        )
    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"❌ Lỗi: {str(e)}"


demo = gr.ChatInterface(
    fn=chat_fn,
    title="📰 Vietnamese News Assistant",
    description=(
        "Hỏi đáp về tin tức tiếng Việt. "
        "Hệ thống tự động tìm bài báo từ database và trả lời dựa trên nội dung thực tế."
    ),
    examples=[
        "Cho tôi biết tin tức về giá vàng hôm nay",
        "Có tin gì mới về công nghệ AI không?",
        "Tình hình kinh tế Việt Nam hiện tại thế nào?",
        "Tin thể thao mới nhất",
        "Tóm tắt tình hình thế giới",
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True)